# L2b: Test and Strengthen a Fibonacci Function

L2a introduced Fibonacci as an example of a function with a documented interface. In this lab, we test that calculation at its numerical boundary, implement a version that prevents silent integer overflow, and write regression tests for its complete interface.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Identify a silent numerical failure:__ Compare a computed result with an exact reference value and explain why successful execution does not establish numerical correctness. Ordinary `Int64` addition wraps when a result cannot be represented, so the calculation for an index just past the supported range finishes quietly and returns a wrong value instead of raising an error.
> * __Implement a defensive numerical interface:__ Validate the input at the function boundary, rejecting non-integers, Booleans, negative indices, and requests beyond index 92 with an `ArgumentError`, and perform every addition with checked arithmetic so an overflow inside the calculation raises an `OverflowError` instead of wrapping. Together, the two checks turn silent overflow into a visible error.
> * __Test the complete function contract:__ Use Julia's `Test` standard library to record the interface as regression tests that verify the ordinary case and its documented indexing convention, both ends of the supported range, and every rejected input category. Rerunning the tests after any change to the implementation either confirms the contract still holds or reports exactly what broke.

Let's get started!
___

## Setup, Data, and Prerequisites

The setup file activates the course environment, loads the student implementation from [`src/Compute.jl`](src/Compute.jl), and imports the packages used in this lab.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [1]:
# Load this lab's file-relative environment, source code, and imports.
include(joinpath(@__DIR__, "Include.jl"));

The setup loads [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/), which provides the testing macros used throughout this lab, and the `L2bFibonacci` module from [`src/Compute.jl`](src/Compute.jl), which exports the `fibonacci_sequence(...)` function you will complete in Task 2.

___

## Task 1: Test the original Fibonacci calculation

The `fibonacci_unchecked(n)` function uses the iterative calculation introduced in L2a. It accepts a nonnegative index and returns a vector containing $F_0$ through $F_n$. The vector stores `Int64` values, but the function does not check whether every requested Fibonacci number fits in that type.

We will test an ordinary case and then inspect the first value outside the `Int64` range.

In [2]:
"""
    fibonacci_unchecked(n::Int64) -> Vector{Int64}

Return the Fibonacci values from F_0 through F_n without checking whether
the requested values fit in Int64.
"""
function fibonacci_unchecked(n::Int64)::Vector{Int64}
    # Reject negative indices before allocating the result vector.
    @assert n >= 0 "n must be nonnegative"

    # Allocate one position for every value from F_0 through F_n.
    sequence = Vector{Int64}(undef, n + 1)
    sequence[1] = 0
    n == 0 && return sequence

    # Store F_1 and compute each remaining value from its two predecessors.
    sequence[2] = 1
    for position in 3:length(sequence)
        sequence[position] = sequence[position - 1] + sequence[position - 2]
    end

    return sequence
end

fibonacci_unchecked

With the function defined, let's evaluate the ordinary case, the boundary case, and the type limit:

In [3]:
# F_10 is an ordinary reference case that fits comfortably in Int64.
observed_F10 = last(fibonacci_unchecked(10))
expected_F10 = 55

# F_93 exceeds typemax(Int64), so unchecked Int64 addition wraps. Use BigInt
# for the exact reference value that the wrapped result should have matched.
observed_F93 = last(fibonacci_unchecked(93))
expected_F93 = big"12200160415121876738"

(
    ordinary_case = (observed = observed_F10, expected = expected_F10),
    boundary_failure = (observed = observed_F93, expected = expected_F93),
    largest_Int64 = typemax(Int64),
)

(ordinary_case = (observed = 55, expected = 55), boundary_failure = (observed = -6246583658587674878, expected = 12200160415121876738), largest_Int64 = 9223372036854775807)

The ordinary case is correct. The call for $F_{93}$ also finishes, but its result is incorrect because the exact value exceeds the largest `Int64` value, which [the `typemax(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.typemax) returns. Ordinary integer addition wraps when the result cannot be represented, so the program returns a value that does not satisfy the function's stated purpose.

A reference case establishes the failure. The type limit identifies the cause. The corrected interface will stop at $F_{92}$, the largest Fibonacci number that fits in an `Int64`.

In [4]:
@testset "original Fibonacci calculation" begin
    # Confirm the ordinary reference case.
    @test observed_F10 == expected_F10

    # Record the known boundary failure before correcting the interface.
    @test observed_F93 != expected_F93
end

Test Summary:                  | Pass  Total  Time
original Fibonacci calculation |    2      2  0.4s


Test.DefaultTestSet("original Fibonacci calculation", Any[], 2, false, false, true, 1.787870926884255e9, 1.787870927307737e9, false, "/Users/jeffreyvarner/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-02/L2b/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X41sZmlsZQ==.jl", Random.Xoshiro(0xeacfdfcbec1c89b2, 0x9a8f2e62fe5955dd, 0x5a5f74c47a32a54e, 0x3a61f580c156f4d6, 0x17086950938677a9))

___

## Task 2: Implement input validation and checked arithmetic

Task 1 showed that the unchecked calculation fails silently: it returns a wrong value instead of raising an error. The corrected interface must validate its input and refuse any request whose exact result cannot be represented in an `Int64`.

[The `fibonacci_sequence(...)` function](src/Compute.jl) takes one integer index `n` and returns the complete sequence $F_0$ through $F_n$. As in the L1d lab, the contract states what the caller supplies and what the function promises to return. A defensive interface adds a third part: the errors it raises instead of returning a wrong value.

> __Input__
>
> * `n::Integer`: the largest Fibonacci index to compute. The supported range is `0 <= n <= 92`. The `Bool` type is a subtype of `Integer` in Julia, but `true` and `false` are not sequence indices, so the function rejects them.
>
> __Output__
>
> * `Vector{Int64}`: the values $F_0$ through $F_n$ in order. Julia arrays use one-based indexing, so $F_n$ is stored at position `n + 1`.
>
> __Errors__
>
> * [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError): the input is not an integer, is a Boolean, is negative, or is greater than 92. Validation raises this error before any calculation begins.
> * [`OverflowError`](https://docs.julialang.org/en/v1/base/base/#Core.OverflowError): an addition inside the calculation exceeded the `Int64` range. The upper bound on `n` should make this impossible, so this error signals an implementation defect rather than a bad input.

The two error types divide the defensive work: input validation guards the boundary of the function, while [the `Base.Checked.checked_add(...)` function](https://docs.julialang.org/en/v1/base/math/#Base.Checked.checked_add) guards every addition inside it by raising an `OverflowError` when a result cannot be represented. With both checks in place, integer overflow raises an error instead of producing a silently wrong value.

Open [`src/Compute.jl`](src/Compute.jl) and complete its three `TODO` sections to implement this contract. Then restart the kernel and run the notebook from the beginning so that Julia loads the revised module. Until every section is complete, an ordinary request such as `fibonacci_sequence(10)` raises a direct implementation error.

Let's recompute the ordinary case from Task 1:

In [5]:
# Compute an ordinary sequence and use position n + 1 to retrieve F_n from
# the one-based vector.
sequence = fibonacci_sequence(10)
(sequence = sequence, F10 = sequence[10 + 1])

ErrorException: The `fibonacci_sequence(...)` function is not implemented yet. Complete TODO 1 through TODO 3.

### Supported and unsupported boundaries

The calls `fibonacci_sequence(0)` and `fibonacci_sequence(1)` are the smallest supported requests. Each must return a complete vector of the documented length. At the other end of the contract, $F_{92}$ is supported and $F_{93}$ is rejected before the calculation begins.

In [6]:
# Evaluate both lower-bound cases and the largest supported result.
base_cases = (n0 = fibonacci_sequence(0), n1 = fibonacci_sequence(1))
largest_supported = last(fibonacci_sequence(92))

# Catch one rejected call so that the error message can be inspected.
upper_boundary_message = try
    fibonacci_sequence(93)
    "no error"
catch error
    sprint(showerror, error)
end

(base_cases = base_cases, F92 = largest_supported, F93 = upper_boundary_message)

ErrorException: The `fibonacci_sequence(...)` function is not implemented yet. Complete TODO 1 through TODO 3.

___

## Task 3: Test the complete interface

Task 2 confirmed the corrected interface by running a few calls and inspecting the results by hand, but nothing repeats that inspection when the implementation changes: a later edit to [`src/Compute.jl`](src/Compute.jl) could reintroduce the kind of silent failure that Task 1 uncovered. So how do we guard against this type of issue?

Regression tests guard against this by recording the behavior that is correct today as executable checks; rerunning them after any change either confirms that the function is still working correctly or reports exactly what is broken.

Julia's [`Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) provides the macros for writing these checks.

| Test tool | Purpose in this lab |
|:--|:--|
| [`@testset`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@testset) | Groups the Fibonacci checks and prints one summary. |
| [`@test`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) | Verifies returned values, vector length, and the upper boundary. |
| [`@test_throws`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test_throws) | Verifies that invalid inputs raise the documented exception type. |

The test set covers the two smallest supported inputs, an ordinary input, the largest supported input, and every input category rejected by the interface.

In [7]:
@testset "defensive Fibonacci interface" begin
    
    # Verify the smallest supported inputs.
    @test fibonacci_sequence(0) == [0]
    @test fibonacci_sequence(1) == [0, 1]

    # Verify an ordinary sequence and its documented indexing convention.
    @test fibonacci_sequence(10) == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
    @test length(fibonacci_sequence(10)) == 11

    # Verify the largest value supported by the Int64 return type.
    @test last(fibonacci_sequence(92)) == 7_540_113_804_746_346_429

    # Verify every invalid-input category in the public contract.
    @test_throws ArgumentError fibonacci_sequence(-1)
    @test_throws ArgumentError fibonacci_sequence(93)
    @test_throws ArgumentError fibonacci_sequence(2.5)
    @test_throws ArgumentError fibonacci_sequence(true)
end

defensive Fibonacci interface: Error During Test at /Users/jeffreyvarner/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-02/L2b/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X52sZmlsZQ==.jl:4
  Test threw exception
  Expression: fibonacci_sequence(0) == [0]
  The `fibonacci_sequence(...)` function is not implemented yet. Complete TODO 1 through TODO 3.
  Stacktrace:
   [1] fibonacci_sequence(n::Int64)
     @ Main.L2bFibonacci ~/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-02/L2b/src/Compute.jl:36
   [2] top-level scope
     @ ~/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks/week-02/L2b/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X52sZmlsZQ==.jl:4
   [3] macro expansion
     @ ~/.julia/juliaup/julia-1.12.7+0.aarch64.apple.darwin14/Julia-1.12.app/Contents/Resources/julia/share/julia/stdlib/v1.12/Test/src/Test.jl:1777 [inlined]
   [4] macro expansion
     @ ~/Desktop/julia_work/CHEME-5800-CourseRepository-Fall-2026/weeks

Test.TestSetException: Some tests did not pass: 1 passed, 3 failed, 5 errored, 0 broken.

### Optional Python implementation

The optional Python implementation in [`src/fibonacci.py`](src/fibonacci.py) uses the same input range and return representation. Python integers do not overflow at $F_{93}$, so the implementation enforces the upper bound explicitly to preserve the shared contract. It also rejects `bool`, which is a subclass of `int` in Python.

The implementation and its tests include comments describing the function and the key steps. You can run the Python regression tests from the repository root with the following command:

```bash
python -m unittest discover -s weeks/week-02/L2b/src -p 'test_*.py'
```

What do we see?

___

## Summary

In this lab, we identified silent integer overflow in a Fibonacci calculation, implemented a guarded `Int64` interface, and wrote regression tests for its supported and unsupported inputs.

> __Key Takeaways:__
>
> * **Successful execution does not establish correctness:** The original calculation returned a value beyond its supported range without raising an error; only comparison with an exact reference value, computed in a type wide enough to hold it, exposed the wrap-around. A reference case establishes that a failure exists, and the type limit identifies its cause.
> * **Validation and checked arithmetic guard different failures:** The `Vector{Int64}` return type determines the supported range, and input validation enforces it by raising an `ArgumentError` before any calculation begins, rejecting non-integers, Booleans, negative indices, and requests past index 92. Checked arithmetic guards every addition inside the calculation, raising an `OverflowError` that signals an implementation defect rather than a bad input, because the validated bound should make internal overflow impossible.
> * **Regression tests record the complete interface:** Tests for the ordinary case, both ends of the supported range, and every rejected input category record the behavior that later changes must preserve. Hand inspection of results happens once, but a test set reruns after every change and fails when the contract breaks.

We will use the same combination of reference cases, explicit numerical limits, input validation, and regression tests when building larger numerical programs.

___